# Day 9 -- NumPy: Reshaping, Math, Stats, Broadcasting (Unit 6 to 12)
### 90-Day Gen AI Engineer Roadmap

---
**Author:** Shaurab Kumar Jha  
**Date:** Day 9 of 90  
**Goal:** MNC-ready Python & Gen AI Engineer

## Topics Covered

| Unit | Topic |
|------|-------|
| Unit 6  | reshape, flatten, ravel, transpose, expand_dims, squeeze |
| Unit 7  | hstack, vstack, concatenate, split -- combining and splitting |
| Unit 8  | Copy vs View -- deep dive (interview favorite) |
| Unit 9  | Arithmetic ops, ufuncs (sqrt, exp, log, clip, abs) |
| Unit 10 | Sigmoid, MSE, BCE from scratch -- no sklearn |
| Unit 11 | Stats -- mean, std, var, argmin, argmax, percentile, corrcoef, cumsum |
| Unit 12 | Broadcasting -- 3 rules, shape compatibility, real examples |

---

**Why this matters for Gen AI:**
- reshape is used every time a tensor goes in or out of a layer
- Broadcasting is how PyTorch/NumPy avoids memory-expensive copies
- MSE/BCE are the actual loss functions you will implement in custom training loops
- corrcoef tells you feature relationships before you even train a model

---

In [1]:
import numpy as np
print(f"NumPy version: {np.__version__}")

NumPy version: 2.3.3


---
# UNIT 6 -- Shape Manipulation
## reshape, flatten, ravel, transpose, expand_dims, squeeze

Shape manipulation is the most common operation in deep learning.
Every time data moves between layers, its shape changes.

Core rule: **total elements must stay the same** before and after reshape.

In [2]:
# --------------------------------------------------
# reshape() -- change shape, total elements same rehne chahiye
# --------------------------------------------------

arr = np.arange(1, 25)   # 24 elements: 1 to 24
print("Original 1D array:", arr)
print("Shape:", arr.shape, "| Size:", arr.size)

# 24 elements ke valid shapes: (24,) (1,24) (24,1) (2,12) (3,8)
# (4,6) (6,4) (8,3) (12,2) (2,3,4) (2,4,3) (3,2,4) (4,3,2) (2,2,6) (2,2,2,3)

r1 = arr.reshape(6, 4)        # 2D: 6 rows, 4 cols
r2 = arr.reshape(2, 3, 4)     # 3D: 2 pages, 3 rows, 4 cols
r3 = arr.reshape(2, 2, 2, 3)  # 4D
r4 = arr.reshape(4, -1)       # -1 means: NumPy calculate karo (24/4 = 6 cols)
r5 = arr.reshape(-1, 8)       # -1 = 24/8 = 3 rows
r6 = arr.reshape(1, -1)       # row vector: (1, 24)
r7 = arr.reshape(-1, 1)       # col vector: (24, 1)

print("\nreshape(6, 4):")
print(r1)
print("\nreshape(2, 3, 4):")
print(r2)
print("\nreshape(4, -1):  shape =", r4.shape, "(-1 auto-calculated to 6)")
print(r4)
print("\nreshape(-1, 8):  shape =", r5.shape)
print(r5)
print("\nreshape(1, -1):  shape =", r6.shape, " <-- row vector")
print("reshape(-1, 1):  shape =", r7.shape, " <-- column vector")
print("4D reshape(2,2,2,3): shape =", r3.shape)

# reshape returns a VIEW (not copy) when possible
arr_base = np.arange(6)
reshaped = arr_base.reshape(2, 3)
reshaped[0, 0] = 999
print("\nVIEW behavior -- reshape modifies original:")
print("Original after reshaped[0,0]=999:", arr_base)
print("(reshape = view, shared memory!)")

# Invalid reshape -- error
try:
    arr.reshape(5, 5)   # 25 != 24
except ValueError as e:
    print(f"\nInvalid reshape error: {e}")

Original 1D array: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24]
Shape: (24,) | Size: 24

reshape(6, 4):
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]
 [17 18 19 20]
 [21 22 23 24]]

reshape(2, 3, 4):
[[[ 1  2  3  4]
  [ 5  6  7  8]
  [ 9 10 11 12]]

 [[13 14 15 16]
  [17 18 19 20]
  [21 22 23 24]]]

reshape(4, -1):  shape = (4, 6) (-1 auto-calculated to 6)
[[ 1  2  3  4  5  6]
 [ 7  8  9 10 11 12]
 [13 14 15 16 17 18]
 [19 20 21 22 23 24]]

reshape(-1, 8):  shape = (3, 8)
[[ 1  2  3  4  5  6  7  8]
 [ 9 10 11 12 13 14 15 16]
 [17 18 19 20 21 22 23 24]]

reshape(1, -1):  shape = (1, 24)  <-- row vector
reshape(-1, 1):  shape = (24, 1)  <-- column vector
4D reshape(2,2,2,3): shape = (2, 2, 2, 3)

VIEW behavior -- reshape modifies original:
Original after reshaped[0,0]=999: [999   1   2   3   4   5]
(reshape = view, shared memory!)

Invalid reshape error: cannot reshape array of size 24 into shape (5,5)


In [3]:
# --------------------------------------------------
# flatten() vs ravel() -- 2D/3D to 1D karna
# Most important difference: COPY vs VIEW
# --------------------------------------------------

mat = np.array([[1, 2, 3],
                [4, 5, 6],
                [7, 8, 9]])
print("Original matrix:")
print(mat)

# flatten() -- hamesha COPY return karta hai
flat = mat.flatten()
print("\nflatten() result:", flat)
flat[0] = 999
print("After flat[0] = 999 :")
print("  flat:     ", flat)
print("  original: ", mat[0])  # unchanged!
print("  (flatten = COPY, original safe)")

# ravel() -- VIEW return karta hai (when possible)
raveled = mat.ravel()
print("\nravel() result:", raveled)
raveled[0] = 777
print("After raveled[0] = 777 :")
print("  raveled:  ", raveled)
print("  original: ", mat[0])   # CHANGED!
print("  (ravel = VIEW, original changed!)")

# flatten order -- C (row-major) vs F (column-major)
mat2 = np.array([[1, 2, 3], [4, 5, 6]])
print("\nflatten order options:")
print("  mat:\n", mat2)
print("  flatten(order='C') row-major:   ", mat2.flatten(order='C'))
print("  flatten(order='F') col-major:   ", mat2.flatten(order='F'))
print("  flatten(order='K') memory order:", mat2.flatten(order='K'))

# Summary
print("\n--- flatten vs ravel summary ---")
print("  flatten() --> COPY  --> safe, use when you dont want original to change")
print("  ravel()   --> VIEW  --> fast, less memory, use when original safe to modify")
print("  Both return 1D array, same result for contiguous arrays")

Original matrix:
[[1 2 3]
 [4 5 6]
 [7 8 9]]

flatten() result: [1 2 3 4 5 6 7 8 9]
After flat[0] = 999 :
  flat:      [999   2   3   4   5   6   7   8   9]
  original:  [1 2 3]
  (flatten = COPY, original safe)

ravel() result: [1 2 3 4 5 6 7 8 9]
After raveled[0] = 777 :
  raveled:   [777   2   3   4   5   6   7   8   9]
  original:  [777   2   3]
  (ravel = VIEW, original changed!)

flatten order options:
  mat:
 [[1 2 3]
 [4 5 6]]
  flatten(order='C') row-major:    [1 2 3 4 5 6]
  flatten(order='F') col-major:    [1 4 2 5 3 6]
  flatten(order='K') memory order: [1 2 3 4 5 6]

--- flatten vs ravel summary ---
  flatten() --> COPY  --> safe, use when you dont want original to change
  ravel()   --> VIEW  --> fast, less memory, use when original safe to modify
  Both return 1D array, same result for contiguous arrays


In [4]:
# --------------------------------------------------
# transpose() and .T -- rows aur columns swap karo
# AI use: weight matrix transpose in backprop,
#         (batch, seq, features) --> (seq, batch, features)
# --------------------------------------------------

# 2D transpose
mat = np.array([[1, 2, 3],
                [4, 5, 6]])
print("Original shape:", mat.shape)
print(mat)

t1 = mat.T            # shorthand
t2 = mat.transpose()  # same thing
print("\nAfter .T -- shape:", t1.shape)
print(t1)
print("mat.T == mat.transpose():", np.array_equal(t1, t2))

# transpose is always a VIEW
t1[0, 0] = 999
print("\nAfter t1[0,0]=999, original mat[0,0]:", mat[0, 0], "<-- changed! (view)")

# 3D transpose -- specify axes order
arr3d = np.arange(24).reshape(2, 3, 4)
print("\n3D array shape:", arr3d.shape, "  (batch=2, rows=3, cols=4)")

# Default transpose -- reverses all axes
t_default = arr3d.transpose()            # (4, 3, 2)
# Custom axis order
t_021 = arr3d.transpose(0, 2, 1)         # (2, 4, 3) swap rows and cols
t_102 = arr3d.transpose(1, 0, 2)         # (3, 2, 4) swap batch and rows
t_210 = arr3d.transpose(2, 1, 0)         # (4, 3, 2) full reverse

print(f"  transpose()     --> shape: {t_default.shape}  (all axes reversed)")
print(f"  transpose(0,2,1)--> shape: {t_021.shape}  (swap last 2 axes)")
print(f"  transpose(1,0,2)--> shape: {t_102.shape}  (swap batch and seq)")
print(f"  transpose(2,1,0)--> shape: {t_210.shape}  (full reverse)")

# Real AI context: NLP tensor
# LLM hidden states: (batch_size, seq_len, hidden_dim) --> (seq_len, batch_size, hidden_dim)
nlp_tensor = np.random.randn(8, 512, 768)  # batch=8, seq=512, hidden=768
swapped = nlp_tensor.transpose(1, 0, 2)    # (seq, batch, hidden)
print(f"\nNLP tensor: {nlp_tensor.shape} --> transpose(1,0,2) --> {swapped.shape}")
print("  (batch, seq, hidden) --> (seq, batch, hidden)")
print("  Used in: attention mechanisms, RNN input formatting")

Original shape: (2, 3)
[[1 2 3]
 [4 5 6]]

After .T -- shape: (3, 2)
[[1 4]
 [2 5]
 [3 6]]
mat.T == mat.transpose(): True

After t1[0,0]=999, original mat[0,0]: 999 <-- changed! (view)

3D array shape: (2, 3, 4)   (batch=2, rows=3, cols=4)
  transpose()     --> shape: (4, 3, 2)  (all axes reversed)
  transpose(0,2,1)--> shape: (2, 4, 3)  (swap last 2 axes)
  transpose(1,0,2)--> shape: (3, 2, 4)  (swap batch and seq)
  transpose(2,1,0)--> shape: (4, 3, 2)  (full reverse)

NLP tensor: (8, 512, 768) --> transpose(1,0,2) --> (512, 8, 768)
  (batch, seq, hidden) --> (seq, batch, hidden)
  Used in: attention mechanisms, RNN input formatting


In [5]:
# --------------------------------------------------
# expand_dims() -- dimension add karo
# squeeze()     -- dimension remove karo (size-1 dimensions)
# Deep learning mein ye dono bahut common hain
# --------------------------------------------------

# expand_dims -- 1 naya axis insert karo
arr = np.array([1, 2, 3, 4, 5])   # shape (5,)
print("Original shape:", arr.shape)

e0 = np.expand_dims(arr, axis=0)   # (1, 5) -- row vector
e1 = np.expand_dims(arr, axis=1)   # (5, 1) -- column vector
e_1 = np.expand_dims(arr, axis=-1) # (5, 1) -- same as axis=1

print("expand_dims(arr, axis=0): shape =", e0.shape, "--", e0)
print("expand_dims(arr, axis=1): shape =", e1.shape, "-- column vector:")
print(e1)

# Equivalent using reshape and None
e_none_0 = arr[np.newaxis, :]   # same as expand_dims axis=0
e_none_1 = arr[:, np.newaxis]   # same as expand_dims axis=1
e_bracket = arr[None, :]        # None is alias for np.newaxis

print("arr[np.newaxis, :] shape:", e_none_0.shape, "-- same as axis=0")
print("arr[:, np.newaxis]  shape:", e_none_1.shape, "-- same as axis=1")
print("arr[None, :]        shape:", e_bracket.shape)

# 2D expand
mat = np.arange(6).reshape(2, 3)   # (2, 3)
m0  = np.expand_dims(mat, axis=0)  # (1, 2, 3) -- add batch dim at front
m1  = np.expand_dims(mat, axis=1)  # (2, 1, 3)
m2  = np.expand_dims(mat, axis=2)  # (2, 3, 1)
print(f"\nmat shape: {mat.shape}")
print(f"expand_dims(mat, 0): {m0.shape}  <-- add batch dim")
print(f"expand_dims(mat, 1): {m1.shape}")
print(f"expand_dims(mat, 2): {m2.shape}")

# Real AI use case:
# Single image (H, W, C) --> batch of 1 image (1, H, W, C)
single_img = np.random.rand(224, 224, 3)  # one image
batch_img = np.expand_dims(single_img, axis=0)  # add batch dim
print(f"\nSingle image: {single_img.shape}")
print(f"After expand_dims: {batch_img.shape}  <-- ready for model.predict()")

# squeeze() -- size-1 dimensions hatao
print("\n--- squeeze() ---")
arr_s = np.array([[[1, 2, 3]]])    # shape (1, 1, 3) -- extra dims
sq_all  = np.squeeze(arr_s)        # all size-1 dims remove --> (3,)
sq_ax0  = np.squeeze(arr_s, axis=0) # only axis 0 remove --> (1, 3)
sq_ax1  = np.squeeze(arr_s, axis=1) # only axis 1 remove --> (1, 3)

print(f"arr_s shape:         {arr_s.shape}")
print(f"squeeze():           {sq_all.shape}   result: {sq_all}")
print(f"squeeze(axis=0):     {sq_ax0.shape}")
print(f"squeeze(axis=1):     {sq_ax1.shape}")

# Model output squeeze
model_out = np.array([[[0.92]]])   # shape (1, 1, 1)
scalar    = np.squeeze(model_out)  # shape () -- scalar
print(f"\nModel output {model_out.shape} --> squeeze --> {scalar.shape}: {scalar}")

Original shape: (5,)
expand_dims(arr, axis=0): shape = (1, 5) -- [[1 2 3 4 5]]
expand_dims(arr, axis=1): shape = (5, 1) -- column vector:
[[1]
 [2]
 [3]
 [4]
 [5]]
arr[np.newaxis, :] shape: (1, 5) -- same as axis=0
arr[:, np.newaxis]  shape: (5, 1) -- same as axis=1
arr[None, :]        shape: (1, 5)

mat shape: (2, 3)
expand_dims(mat, 0): (1, 2, 3)  <-- add batch dim
expand_dims(mat, 1): (2, 1, 3)
expand_dims(mat, 2): (2, 3, 1)

Single image: (224, 224, 3)
After expand_dims: (1, 224, 224, 3)  <-- ready for model.predict()

--- squeeze() ---
arr_s shape:         (1, 1, 3)
squeeze():           (3,)   result: [1 2 3]
squeeze(axis=0):     (1, 3)
squeeze(axis=1):     (1, 3)

Model output (1, 1, 1) --> squeeze --> (): 0.92


---
# UNIT 7 -- Combining and Splitting Arrays
## hstack, vstack, concatenate, split, hsplit, vsplit

In [6]:
# --------------------------------------------------
# np.concatenate() -- most general join function
# axis specify karo -- default axis=0
# --------------------------------------------------

a = np.array([[1, 2, 3],
              [4, 5, 6]])

b = np.array([[7,  8,  9],
              [10, 11, 12]])

# axis=0 -- rows ke saath join (vertically)
c0 = np.concatenate([a, b], axis=0)
print("a shape:", a.shape, "  b shape:", b.shape)
print("\nconcatenate axis=0 (vertical join) shape:", c0.shape)
print(c0)

# axis=1 -- columns ke saath join (horizontally)
c1 = np.concatenate([a, b], axis=1)
print("\nconcatenate axis=1 (horizontal join) shape:", c1.shape)
print(c1)

# Multiple arrays concatenate
x = np.array([[1, 2], [3, 4]])
y = np.array([[5, 6], [7, 8]])
z = np.array([[9, 10], [11, 12]])
multi = np.concatenate([x, y, z], axis=0)
print("\n3 arrays concatenated (axis=0) shape:", multi.shape)
print(multi)

a shape: (2, 3)   b shape: (2, 3)

concatenate axis=0 (vertical join) shape: (4, 3)
[[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]

concatenate axis=1 (horizontal join) shape: (2, 6)
[[ 1  2  3  7  8  9]
 [ 4  5  6 10 11 12]]

3 arrays concatenated (axis=0) shape: (6, 2)
[[ 1  2]
 [ 3  4]
 [ 5  6]
 [ 7  8]
 [ 9 10]
 [11 12]]


In [7]:
# --------------------------------------------------
# vstack, hstack, dstack -- shortcuts for common joins
# --------------------------------------------------

a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

# vstack -- vertical stack (rows add karo) = concatenate axis=0
vs = np.vstack([a, b])
print("vstack (vertical, add rows) shape:", vs.shape)
print(vs)

# hstack -- horizontal stack (columns add karo) = concatenate axis=1
hs = np.hstack([a, b])
print("\nhstack (horizontal, add cols) shape:", hs.shape)
print(hs)

# 1D arrays ke saath
p = np.array([1, 2, 3])
q = np.array([4, 5, 6])
print("\n1D vstack:")
print(np.vstack([p, q]))      # becomes 2D: (2, 3)
print("1D hstack:", np.hstack([p, q]))  # stays 1D: (6,)

# dstack -- depth stack (axis=2)
ds = np.dstack([a, b])   # stacks along third axis
print("\ndstack (depth, axis=2) shape:", ds.shape)
print(ds)

# np.stack() -- NEW axis create karke stack karo (different from concat!)
stacked_0 = np.stack([a, b], axis=0)  # (2, 2, 2) -- new axis at 0
stacked_1 = np.stack([a, b], axis=1)  # (2, 2, 2) -- new axis at 1
stacked_2 = np.stack([a, b], axis=2)  # (2, 2, 2) -- new axis at 2

print("\nnp.stack vs np.concatenate:")
print(f"  concatenate axis=0: {np.concatenate([a,b], axis=0).shape} -- no new axis")
print(f"  stack axis=0:       {stacked_0.shape} -- NEW axis created")
print(f"  stack axis=1:       {stacked_1.shape} -- NEW axis created")
print(f"  stack axis=2:       {stacked_2.shape} -- NEW axis created")
print("Rule: concatenate = join along existing axis, stack = create new axis")

# Real AI use case: batch of embeddings
emb1 = np.random.randn(768)  # one embedding vector
emb2 = np.random.randn(768)
emb3 = np.random.randn(768)
batch = np.stack([emb1, emb2, emb3], axis=0)  # (3, 768) batch
print(f"\nEmbedding stack: 3 x {emb1.shape} --> {batch.shape}  (batch ready)")

vstack (vertical, add rows) shape: (4, 2)
[[1 2]
 [3 4]
 [5 6]
 [7 8]]

hstack (horizontal, add cols) shape: (2, 4)
[[1 2 5 6]
 [3 4 7 8]]

1D vstack:
[[1 2 3]
 [4 5 6]]
1D hstack: [1 2 3 4 5 6]

dstack (depth, axis=2) shape: (2, 2, 2)
[[[1 5]
  [2 6]]

 [[3 7]
  [4 8]]]

np.stack vs np.concatenate:
  concatenate axis=0: (4, 2) -- no new axis
  stack axis=0:       (2, 2, 2) -- NEW axis created
  stack axis=1:       (2, 2, 2) -- NEW axis created
  stack axis=2:       (2, 2, 2) -- NEW axis created
Rule: concatenate = join along existing axis, stack = create new axis

Embedding stack: 3 x (768,) --> (3, 768)  (batch ready)


In [8]:
# --------------------------------------------------
# Splitting -- np.split, hsplit, vsplit, array_split
# AI use: train/val/test split, batch generation
# --------------------------------------------------

arr = np.arange(1, 13)  # [1, 2, ..., 12]
print("Original array:", arr)

# np.split(arr, n_splits) -- equal splits only
s_equal = np.split(arr, 3)    # 12/3 = 4 each
s_idx   = np.split(arr, [3, 7, 10])  # split at indices 3, 7, 10

print("\nnp.split(arr, 3) -- 3 equal parts:")
for i, part in enumerate(s_equal):
    print(f"  Part {i}: {part}")

print("\nnp.split(arr, [3,7,10]) -- at index positions:")
for i, part in enumerate(s_idx):
    print(f"  Part {i}: {part}")

# array_split -- unequal splits allowed
unequal = np.array_split(arr, 5)  # 12/5 is not integer, ok
print("\nnp.array_split(arr, 5) -- unequal ok:")
for i, part in enumerate(unequal):
    print(f"  Part {i}: {part} (size {len(part)})")

# 2D splitting
mat = np.arange(1, 25).reshape(4, 6)
print("\n2D matrix (4x6):")
print(mat)

# vsplit -- horizontal lines se kaato (axis=0, rows split)
top, bottom = np.vsplit(mat, 2)   # 4 rows / 2 = 2 each
print("\nvsplit into 2 (top 2 rows, bottom 2 rows):")
print("Top:   "); print(top)
print("Bottom:"); print(bottom)

# hsplit -- vertical lines se kaato (axis=1, cols split)
left, mid, right = np.hsplit(mat, 3)  # 6 cols / 3 = 2 each
print("\nhsplit into 3 (cols split):")
print("Left: "); print(left)
print("Mid:  "); print(mid)
print("Right:"); print(right)

# Real use: train/val/test split manually
data = np.random.randn(100, 5)  # 100 samples, 5 features
train, val, test = np.split(data, [70, 85])  # 70 train, 15 val, 15 test
print(f"\nDataset split: train={train.shape}, val={val.shape}, test={test.shape}")

Original array: [ 1  2  3  4  5  6  7  8  9 10 11 12]

np.split(arr, 3) -- 3 equal parts:
  Part 0: [1 2 3 4]
  Part 1: [5 6 7 8]
  Part 2: [ 9 10 11 12]

np.split(arr, [3,7,10]) -- at index positions:
  Part 0: [1 2 3]
  Part 1: [4 5 6 7]
  Part 2: [ 8  9 10]
  Part 3: [11 12]

np.array_split(arr, 5) -- unequal ok:
  Part 0: [1 2 3] (size 3)
  Part 1: [4 5 6] (size 3)
  Part 2: [7 8] (size 2)
  Part 3: [ 9 10] (size 2)
  Part 4: [11 12] (size 2)

2D matrix (4x6):
[[ 1  2  3  4  5  6]
 [ 7  8  9 10 11 12]
 [13 14 15 16 17 18]
 [19 20 21 22 23 24]]

vsplit into 2 (top 2 rows, bottom 2 rows):
Top:   
[[ 1  2  3  4  5  6]
 [ 7  8  9 10 11 12]]
Bottom:
[[13 14 15 16 17 18]
 [19 20 21 22 23 24]]

hsplit into 3 (cols split):
Left: 
[[ 1  2]
 [ 7  8]
 [13 14]
 [19 20]]
Mid:  
[[ 3  4]
 [ 9 10]
 [15 16]
 [21 22]]
Right:
[[ 5  6]
 [11 12]
 [17 18]
 [23 24]]

Dataset split: train=(70, 5), val=(15, 5), test=(15, 5)


---
# UNIT 8 -- Copy vs View (Deep Dive)

This is the most common interview question on NumPy.

**Three levels:**
- **No copy** -- same object, same data
- **View (shallow copy)** -- different object, same data block in memory
- **Deep copy** -- completely independent, different memory

In [9]:
# --------------------------------------------------
# LEVEL 1: No copy -- simple assignment
# --------------------------------------------------

original = np.array([1, 2, 3, 4, 5])
alias = original    # NO copy -- same object!

print("--- LEVEL 1: No Copy (simple assignment) ---")
print(f"original id: {id(original)}")
print(f"alias id:    {id(alias)}")
print(f"Same object? {original is alias}")

alias[0] = 999
print(f"After alias[0]=999:")
print(f"  alias:    {alias}")
print(f"  original: {original}  <-- changed! same object")

# shape change on alias changes original too
alias.shape = (5,)
print(f"original.shape: {original.shape}  <-- alias.shape change affects original")

--- LEVEL 1: No Copy (simple assignment) ---
original id: 2860788730256
alias id:    2860788730256
Same object? True
After alias[0]=999:
  alias:    [999   2   3   4   5]
  original: [999   2   3   4   5]  <-- changed! same object
original.shape: (5,)  <-- alias.shape change affects original


In [10]:
# --------------------------------------------------
# LEVEL 2: View (shallow copy) -- different object, same memory
# Operations that create views:
#   slicing, reshape, transpose, ravel, view(), flatten(order='F')
#   expand_dims, squeeze, np.broadcast_to
# --------------------------------------------------

print("--- LEVEL 2: View ---")

original = np.array([10, 20, 30, 40, 50, 60])

view_slice  = original[1:4]         # slicing = view
view_ravel  = original.ravel()       # ravel = view (for contiguous)
view_reshape = original.reshape(2, 3) # reshape = view (usually)
view_T      = original.reshape(2, 3).T  # transpose = view

print(f"original:            {original}")
print(f"view_slice [1:4]:     {view_slice}")

# Check if view with .base attribute
# .base is None means it owns its data (not a view)
# .base is not None means it is a view of something
print(f"\n.base attribute check (None = owns data, not-None = view):")
print(f"  original.base:     {original.base}   <-- owns its data")
print(f"  view_slice.base:   is original? {view_slice.base is original}  <-- view!")
print(f"  view_reshape.base: is original? {view_reshape.base is original}")

# Modifying view modifies original
view_slice[0] = 777
print(f"\nAfter view_slice[0]=777:")
print(f"  view_slice: {view_slice}")
print(f"  original:   {original}  <-- index 1 changed!")

# shape change on view does NOT change original shape
view_reshape.shape = (3, 2)   # only view shape changes
print(f"\nview_reshape after .shape=(3,2): {view_reshape.shape}")
print(f"original shape still: {original.shape}  <-- shape changes don't propagate")

# np.may_share_memory() -- check karo
print(f"\nnp.may_share_memory:")
print(f"  original & view_slice:  {np.may_share_memory(original, view_slice)}")
print(f"  original & view_reshape:{np.may_share_memory(original, view_reshape)}")

# np.shares_memory() -- strict check
print(f"  np.shares_memory (strict):  {np.shares_memory(original, view_slice)}")

--- LEVEL 2: View ---
original:            [10 20 30 40 50 60]
view_slice [1:4]:     [20 30 40]

.base attribute check (None = owns data, not-None = view):
  original.base:     None   <-- owns its data
  view_slice.base:   is original? True  <-- view!
  view_reshape.base: is original? True

After view_slice[0]=777:
  view_slice: [777  30  40]
  original:   [ 10 777  30  40  50  60]  <-- index 1 changed!

view_reshape after .shape=(3,2): (3, 2)
original shape still: (6,)  <-- shape changes don't propagate

np.may_share_memory:
  original & view_slice:  True
  original & view_reshape:True
  np.shares_memory (strict):  True


In [11]:
# --------------------------------------------------
# LEVEL 3: Deep copy -- completely independent
# Use .copy() or np.copy()
# --------------------------------------------------

print("--- LEVEL 3: Deep Copy ---")

original = np.array([1, 2, 3, 4, 5])

deep_copy = original.copy()          # method 1
deep_copy2 = np.copy(original)       # method 2
deep_copy3 = original[::1].copy()    # slice then copy

print(f"original:   {original}")
print(f"deep_copy:  {deep_copy}")
print(f"Same object? {original is deep_copy}")
print(f"Shares memory? {np.shares_memory(original, deep_copy)}")

deep_copy[0] = 999
print(f"\nAfter deep_copy[0]=999:")
print(f"  deep_copy: {deep_copy}")
print(f"  original:  {original}  <-- UNCHANGED!")

# --------------------------------------------------
# Which operations give VIEW vs COPY?
# --------------------------------------------------

print("\n=== VIEW vs COPY -- Complete Reference ===")
arr = np.array([1, 2, 3, 4, 5, 6])

checks = [
    ("arr[1:4]       (slicing)",     arr[1:4]),
    ("arr.reshape(2,3)",              arr.reshape(2, 3)),
    ("arr.ravel()",                   arr.ravel()),
    ("arr.T",                         arr.reshape(2,3).T),
    ("arr.view(np.float64)",          arr.view(np.float32)),
    ("arr.flatten()   (COPY!)",       arr.flatten()),
    ("arr.copy()      (COPY!)",       arr.copy()),
    ("arr[[0,1,2]]   fancy (COPY!)",  arr[[0,1,2]]),
    ("arr[arr>3]     bool  (COPY!)",  arr[arr>3]),
    ("np.squeeze",                    np.expand_dims(arr, 0).squeeze()),
]

print(f"  {'Operation':<35} {'Type'}")
print("  " + "-" * 50)
for name, result in checks:
    is_view = result.base is not None
    type_str = "VIEW  (shares memory)" if is_view else "COPY  (independent)"
    print(f"  {name:<35} {type_str}")

print("\nKey Rule:")
print("  Basic slicing   --> VIEW")
print("  Boolean index   --> COPY")
print("  Fancy index     --> COPY")
print("  reshape/ravel   --> VIEW (when memory is contiguous)")
print("  flatten         --> always COPY")
print("  .copy()         --> always COPY")

--- LEVEL 3: Deep Copy ---
original:   [1 2 3 4 5]
deep_copy:  [1 2 3 4 5]
Same object? False
Shares memory? False

After deep_copy[0]=999:
  deep_copy: [999   2   3   4   5]
  original:  [1 2 3 4 5]  <-- UNCHANGED!

=== VIEW vs COPY -- Complete Reference ===
  Operation                           Type
  --------------------------------------------------
  arr[1:4]       (slicing)            VIEW  (shares memory)
  arr.reshape(2,3)                    VIEW  (shares memory)
  arr.ravel()                         VIEW  (shares memory)
  arr.T                               VIEW  (shares memory)
  arr.view(np.float64)                VIEW  (shares memory)
  arr.flatten()   (COPY!)             COPY  (independent)
  arr.copy()      (COPY!)             COPY  (independent)
  arr[[0,1,2]]   fancy (COPY!)        COPY  (independent)
  arr[arr>3]     bool  (COPY!)        COPY  (independent)
  np.squeeze                          VIEW  (shares memory)

Key Rule:
  Basic slicing   --> VIEW
  Boolean inde

---
# UNIT 9 -- Arithmetic Operations and Universal Functions (ufuncs)

**ufunc = Universal Function** -- operates element-wise on ndarrays.

NumPy has 60+ ufuncs. They are C-compiled, vectorized, and support broadcasting.

In [12]:
# --------------------------------------------------
# ARITHMETIC OPERATIONS -- element-wise by default
# --------------------------------------------------

a = np.array([10, 20, 30, 40, 50])
b = np.array([ 2,  4,  6,  8, 10])

print("a:", a)
print("b:", b)
print()

# Operator form vs function form -- both identical
print("Addition:        a + b   = ", a + b,    " | np.add(a,b)      = ", np.add(a, b))
print("Subtraction:     a - b   = ", a - b,    " | np.subtract(a,b) = ", np.subtract(a, b))
print("Multiplication:  a * b   = ", a * b,    " | np.multiply(a,b) = ", np.multiply(a, b))
print("Division:        a / b   = ", a / b,    " | np.divide(a,b)   = ", np.divide(a, b))
print("Floor division:  a // b  = ", a // b,   " | np.floor_divide  = ", np.floor_divide(a,b))
print("Modulo:          a % b   = ", a % b,    " | np.mod(a,b)      = ", np.mod(a, b))
print("Power:           a ** 2  = ", a ** 2,   " | np.power(a,2)    = ", np.power(a, 2))

# Scalar arithmetic -- broadcasts to all elements
print("\nScalar ops (broadcast to all elements):")
print("a + 100:  ", a + 100)
print("a * 0.1:  ", a * 0.1)
print("a ** 0.5: ", a ** 0.5)
print("100 / a:  ", 100 / a)

# Comparison operators -- return boolean array
print("\nComparison (element-wise, returns bool array):")
print("a > 25:    ", a > 25)
print("a == 30:   ", a == 30)
print("a != b*5:  ", a != b * 5)
print("a >= b*5:  ", a >= b * 5)

a: [10 20 30 40 50]
b: [ 2  4  6  8 10]

Addition:        a + b   =  [12 24 36 48 60]  | np.add(a,b)      =  [12 24 36 48 60]
Subtraction:     a - b   =  [ 8 16 24 32 40]  | np.subtract(a,b) =  [ 8 16 24 32 40]
Multiplication:  a * b   =  [ 20  80 180 320 500]  | np.multiply(a,b) =  [ 20  80 180 320 500]
Division:        a / b   =  [5. 5. 5. 5. 5.]  | np.divide(a,b)   =  [5. 5. 5. 5. 5.]
Floor division:  a // b  =  [5 5 5 5 5]  | np.floor_divide  =  [5 5 5 5 5]
Modulo:          a % b   =  [0 0 0 0 0]  | np.mod(a,b)      =  [0 0 0 0 0]
Power:           a ** 2  =  [ 100  400  900 1600 2500]  | np.power(a,2)    =  [ 100  400  900 1600 2500]

Scalar ops (broadcast to all elements):
a + 100:   [110 120 130 140 150]
a * 0.1:   [1. 2. 3. 4. 5.]
a ** 0.5:  [3.16227766 4.47213595 5.47722558 6.32455532 7.07106781]
100 / a:   [10.          5.          3.33333333  2.5         2.        ]

Comparison (element-wise, returns bool array):
a > 25:     [False False  True  True  True]
a == 30:    [False 

In [13]:
# --------------------------------------------------
# MATH UFUNCS -- sqrt, exp, log, abs, sign, floor, ceil, round
# --------------------------------------------------

arr = np.array([0.25, 1, 4, 9, 16, 25, 100])
print("Array:", arr)

print("\nnp.sqrt():       ", np.sqrt(arr))         # square root
print("np.cbrt():       ", np.round(np.cbrt(arr), 3))  # cube root
print("np.square():     ", np.square(arr[:4]))    # x^2
print("np.abs():        ", np.abs(np.array([-3, 4, -7, 0, -1])))

# Exponential family
x = np.array([0, 1, 2, 3, -1, -2])
print("\nExponential family:")
print("  np.exp(x):      ", np.round(np.exp(x), 4))     # e^x
print("  np.exp2(x):     ", np.round(np.exp2(x), 4))    # 2^x
print("  np.expm1(x):    ", np.round(np.expm1(x), 4))   # e^x - 1 (precise for small x)

# Log family
pos = np.array([0.01, 0.1, 1, np.e, 10, 100])
print("\nLogarithm family:")
print("  np.log(x):      ", np.round(np.log(pos), 3))    # natural log ln(x)
print("  np.log2(x):     ", np.round(np.log2(pos), 3))   # log base 2
print("  np.log10(x):    ", np.round(np.log10(pos), 3))  # log base 10
print("  np.log1p(x):    ", np.round(np.log1p(pos[:4]), 4)) # log(1+x) precise

# Rounding family
r = np.array([-2.7, -1.5, -0.3, 0.5, 1.2, 2.9])
print("\nRounding family:")
print("  np.floor():  ", np.floor(r))    # floor: always down
print("  np.ceil():   ", np.ceil(r))     # ceil:  always up
print("  np.round():  ", np.round(r))    # round: nearest even (banker's rounding)
print("  np.trunc():  ", np.trunc(r))    # truncate toward zero
print("  np.rint():   ", np.rint(r))     # round to nearest int

# Sign
s = np.array([-5, -1, 0, 1, 5])
print("\nnp.sign():       ", np.sign(s))  # -1, 0, or +1

Array: [  0.25   1.     4.     9.    16.    25.   100.  ]

np.sqrt():        [ 0.5  1.   2.   3.   4.   5.  10. ]
np.cbrt():        [0.63  1.    1.587 2.08  2.52  2.924 4.642]
np.square():      [6.25e-02 1.00e+00 1.60e+01 8.10e+01]
np.abs():         [3 4 7 0 1]

Exponential family:
  np.exp(x):       [ 1.      2.7183  7.3891 20.0855  0.3679  0.1353]
  np.exp2(x):      [1.   2.   4.   8.   0.5  0.25]
  np.expm1(x):     [ 0.      1.7183  6.3891 19.0855 -0.6321 -0.8647]

Logarithm family:
  np.log(x):       [-4.605 -2.303  0.     1.     2.303  4.605]
  np.log2(x):      [-6.644 -3.322  0.     1.443  3.322  6.644]
  np.log10(x):     [-2.    -1.     0.     0.434  1.     2.   ]
  np.log1p(x):     [0.01   0.0953 0.6931 1.3133]

Rounding family:
  np.floor():   [-3. -2. -1.  0.  1.  2.]
  np.ceil():    [-2. -1. -0.  1.  2.  3.]
  np.round():   [-3. -2. -0.  0.  1.  3.]
  np.trunc():   [-2. -1. -0.  0.  1.  2.]
  np.rint():    [-3. -2. -0.  0.  1.  3.]

np.sign():        [-1 -1  0  1  1]


In [14]:
# --------------------------------------------------
# np.clip() -- values ko range mein rokna
# AI use: gradient clipping, pixel clamping, probability bounds
# --------------------------------------------------

arr = np.array([-10, -3, 0, 5, 12, 25, 100, -50])
print("Original:          ", arr)

clipped = np.clip(arr, -5, 20)
print("clip(-5, 20):      ", clipped)
print("  (below -5 becomes -5, above 20 becomes 20)")

# Clip probabilities to avoid log(0) = -inf
probs = np.array([0.0, 0.001, 0.5, 0.999, 1.0])
epsilon = 1e-7
safe_probs = np.clip(probs, epsilon, 1 - epsilon)
print("\nProb clipping (avoid log(0)):")
print("  Original:  ", probs)
print("  Clipped:   ", safe_probs)
print("  log values:", np.round(np.log(safe_probs), 3), "  <-- no -inf!")

# Gradient clipping -- prevent exploding gradients
gradients = np.array([0.01, -0.02, 5.3, -8.1, 0.001, 12.5])
max_norm  = 1.0
clipped_grads = np.clip(gradients, -max_norm, max_norm)
print("\nGradient clipping (max_norm=1.0):")
print("  Gradients:       ", gradients)
print("  Clipped grads:   ", clipped_grads)

# Image pixel clamping
pixels = np.array([-10, 0, 128, 255, 300, 1000])
valid_pixels = np.clip(pixels, 0, 255)
print("\nImage pixel clamping [0, 255]:")
print("  Original: ", pixels)
print("  Clamped:  ", valid_pixels)

Original:           [-10  -3   0   5  12  25 100 -50]
clip(-5, 20):       [-5 -3  0  5 12 20 20 -5]
  (below -5 becomes -5, above 20 becomes 20)

Prob clipping (avoid log(0)):
  Original:   [0.    0.001 0.5   0.999 1.   ]
  Clipped:    [1.000000e-07 1.000000e-03 5.000000e-01 9.990000e-01 9.999999e-01]
  log values: [-1.6118e+01 -6.9080e+00 -6.9300e-01 -1.0000e-03 -0.0000e+00]   <-- no -inf!

Gradient clipping (max_norm=1.0):
  Gradients:        [ 1.00e-02 -2.00e-02  5.30e+00 -8.10e+00  1.00e-03  1.25e+01]
  Clipped grads:    [ 0.01  -0.02   1.    -1.     0.001  1.   ]

Image pixel clamping [0, 255]:
  Original:  [ -10    0  128  255  300 1000]
  Clamped:   [  0   0 128 255 255 255]


---
# UNIT 10 -- ML Functions from Scratch (No Sklearn)

## Sigmoid, Softmax, MSE, MAE, RMSE, R2, Binary Cross Entropy

Yeh portfolio ke liye gold hai. Recruiters dekhte hain ki tum ML math samajhte ho, sirf library call nahi karte.

In [15]:
# --------------------------------------------------
# ACTIVATION FUNCTIONS -- from scratch
# --------------------------------------------------

# --- Sigmoid ---
# Formula: sigma(x) = 1 / (1 + e^(-x))
# Output range: (0, 1)
# Use: binary classification output layer

def sigmoid(x):
    """
    Sigmoid activation function.
    Input: any real number or array
    Output: value between 0 and 1
    """
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_stable(x):
    """
    Numerically stable sigmoid.
    Avoids exp overflow for large negative x.
    """
    return np.where(
        x >= 0,
        1.0 / (1.0 + np.exp(-x)),
        np.exp(x) / (1.0 + np.exp(x))
    )

def sigmoid_derivative(x):
    """
    Derivative of sigmoid: sigma(x) * (1 - sigma(x))
    Used in backpropagation.
    """
    s = sigmoid(x)
    return s * (1 - s)

x = np.array([-10, -5, -2, -1, 0, 1, 2, 5, 10])
print("=" * 55)
print("SIGMOID")
print("=" * 55)
print(f"Formula: sigma(x) = 1 / (1 + e^(-x))")
print(f"Output range: (0, 1)\n")
print(f"{'x':>8}   {'sigmoid':>12}   {'derivative':>12}")
print("-" * 38)
for xi in x:
    s  = sigmoid(xi)
    ds = sigmoid_derivative(xi)
    print(f"  {xi:>5}    {s:>12.6f}    {ds:>12.6f}")

# Key values to remember
print("\nKey values:")
print(f"  sigmoid(0)   = {sigmoid(0):.4f}  (middle -- threshold for binary classification)")
print(f"  sigmoid(+inf) ~= {sigmoid(1000):.4f}  (approaches 1)")
print(f"  sigmoid(-inf) ~= {sigmoid(-1000):.6f}  (approaches 0)")
print(f"  sigmoid(1)   = {sigmoid(1):.4f}")
print(f"  sigmoid(-1)  = {sigmoid(-1):.4f}")

SIGMOID
Formula: sigma(x) = 1 / (1 + e^(-x))
Output range: (0, 1)

       x        sigmoid     derivative
--------------------------------------
    -10        0.000045        0.000045
     -5        0.006693        0.006648
     -2        0.119203        0.104994
     -1        0.268941        0.196612
      0        0.500000        0.250000
      1        0.731059        0.196612
      2        0.880797        0.104994
      5        0.993307        0.006648
     10        0.999955        0.000045

Key values:
  sigmoid(0)   = 0.5000  (middle -- threshold for binary classification)
  sigmoid(+inf) ~= 1.0000  (approaches 1)
  sigmoid(-inf) ~= 0.000000  (approaches 0)
  sigmoid(1)   = 0.7311
  sigmoid(-1)  = 0.2689


C:\Users\shaur\AppData\Local\Temp\ipykernel_9488\1965682656.py:16: RuntimeWarning: overflow encountered in exp
  return 1.0 / (1.0 + np.exp(-x))


In [16]:
# --- ReLU and variants ---

def relu(x):
    """ReLU: max(0, x)"""
    return np.maximum(0, x)

def relu_derivative(x):
    """ReLU derivative: 1 if x>0, else 0"""
    return (x > 0).astype(float)

def leaky_relu(x, alpha=0.01):
    """Leaky ReLU: x if x>0, else alpha*x"""
    return np.where(x > 0, x, alpha * x)

def elu(x, alpha=1.0):
    """ELU: x if x>0, else alpha*(e^x - 1)"""
    return np.where(x > 0, x, alpha * (np.exp(x) - 1))

def gelu(x):
    """GELU: used in BERT, GPT. Gaussian Error Linear Unit."""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

def softmax(x):
    """
    Softmax: converts logits to probabilities.
    Subtract max for numerical stability (avoids exp overflow).
    Output: probabilities that sum to 1.
    Use: multi-class output layer.
    """
    x_shifted = x - np.max(x)   # stability trick
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x)

x = np.array([-3.0, -1.0, 0.0, 1.0, 2.0, 3.0])

print("Activation Comparison:")
print(f"{'x':>6}  {'ReLU':>8}  {'L-ReLU':>8}  {'ELU':>8}  {'GELU':>8}  {'tanh':>8}")
print("-" * 55)
for xi in x:
    print(f"  {xi:>4.1f}  {relu(xi):>8.4f}  {leaky_relu(xi):>8.4f}  "
          f"{elu(xi):>8.4f}  {gelu(xi):>8.4f}  {np.tanh(xi):>8.4f}")

# Softmax demo
logits = np.array([2.0, 1.0, 0.5, -1.0, 3.0])
probs  = softmax(logits)
print("\nSoftmax (multi-class):")
print(f"  Logits:  {logits}")
print(f"  Probs:   {np.round(probs, 4)}")
print(f"  Sum:     {probs.sum():.6f}  (must be 1.0)")
print(f"  argmax:  {np.argmax(probs)}  (predicted class)")

Activation Comparison:
     x      ReLU    L-ReLU       ELU      GELU      tanh
-------------------------------------------------------
  -3.0    0.0000   -0.0300   -0.9502   -0.0036   -0.9951
  -1.0    0.0000   -0.0100   -0.6321   -0.1588   -0.7616
   0.0    0.0000    0.0000    0.0000    0.0000    0.0000
   1.0    1.0000    1.0000    1.0000    0.8412    0.7616
   2.0    2.0000    2.0000    2.0000    1.9546    0.9640
   3.0    3.0000    3.0000    3.0000    2.9964    0.9951

Softmax (multi-class):
  Logits:  [ 2.   1.   0.5 -1.   3. ]
  Probs:   [0.2294 0.0844 0.0512 0.0114 0.6236]
  Sum:     1.000000  (must be 1.0)
  argmax:  4  (predicted class)


In [17]:
# --------------------------------------------------
# REGRESSION LOSS FUNCTIONS from scratch
# MSE, MAE, RMSE, R-squared
# --------------------------------------------------

def mse(y_true, y_pred):
    """
    Mean Squared Error.
    Formula: (1/n) * sum((y_true - y_pred)^2)
    Properties:
      - penalizes large errors heavily (squared)
      - differentiable everywhere (good for gradient descent)
      - same units as y^2 (harder to interpret)
    """
    return np.mean((y_true - y_pred) ** 2)

def mse_gradient(y_true, y_pred):
    """
    Gradient of MSE w.r.t. y_pred.
    dL/dy_pred = (-2/n) * (y_true - y_pred)
    Used in backprop.
    """
    n = len(y_true)
    return (-2 / n) * (y_true - y_pred)

def mae(y_true, y_pred):
    """
    Mean Absolute Error.
    Formula: (1/n) * sum(|y_true - y_pred|)
    Properties:
      - robust to outliers (linear, not squared)
      - same units as y (interpretable)
      - not differentiable at 0 (needs subgradient)
    """
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    """
    Root Mean Squared Error.
    Formula: sqrt(MSE)
    Properties: same units as y, more interpretable than MSE
    """
    return np.sqrt(mse(y_true, y_pred))

def r_squared(y_true, y_pred):
    """
    R-squared (Coefficient of Determination).
    Formula: 1 - SS_res / SS_tot
    SS_res = sum((y_true - y_pred)^2)
    SS_tot = sum((y_true - mean(y_true))^2)
    Range: (-inf, 1.0]
      1.0  = perfect predictions
      0.0  = model predicts mean, no better than baseline
      < 0  = model is worse than predicting the mean
    """
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def huber_loss(y_true, y_pred, delta=1.0):
    """
    Huber Loss -- combines MSE and MAE.
    For |error| <= delta: 0.5 * error^2   (MSE behavior)
    For |error|  > delta: delta * (|error| - 0.5*delta)  (MAE behavior)
    Best of both: smooth near zero, robust to outliers.
    """
    error = np.abs(y_true - y_pred)
    return np.mean(np.where(
        error <= delta,
        0.5 * error ** 2,
        delta * (error - 0.5 * delta)
    ))


# Test with house price prediction example
y_true = np.array([250, 300, 180, 420, 350, 280, 190, 500], dtype=float)  # actual prices (thousands)
y_pred = np.array([270, 290, 200, 400, 380, 260, 210, 480], dtype=float)  # predicted prices

print("=" * 55)
print("REGRESSION LOSSES -- House Price Prediction")
print("=" * 55)
print(f"  y_true: {y_true}")
print(f"  y_pred: {y_pred}")
print(f"  Errors: {y_pred - y_true}")
print()
print(f"  MSE:        {mse(y_true, y_pred):.4f}  (penalizes large errors heavily)")
print(f"  RMSE:       {rmse(y_true, y_pred):.4f}  (interpretable: avg error in same units)")
print(f"  MAE:        {mae(y_true, y_pred):.4f}  (avg absolute error, robust to outliers)")
print(f"  R-squared:  {r_squared(y_true, y_pred):.4f}  (1.0 = perfect, 0.0 = predicts mean)")
print(f"  Huber:      {huber_loss(y_true, y_pred):.4f}")

# Outlier effect comparison
y_true_out = y_true.copy()
y_pred_out = y_pred.copy()
y_pred_out[0] = 600   # big outlier: true=250, pred=600

print("\nOutlier effect (one prediction off by 350):")
print(f"  MSE  before: {mse(y_true, y_pred):.2f}  | after: {mse(y_true_out, y_pred_out):.2f}  (MSE very sensitive!)")
print(f"  MAE  before: {mae(y_true, y_pred):.2f}  | after: {mae(y_true_out, y_pred_out):.2f}  (MAE robust)")
print(f"  RMSE before: {rmse(y_true, y_pred):.2f}  | after: {rmse(y_true_out, y_pred_out):.2f}")
print(f"  Huber before:{huber_loss(y_true, y_pred):.2f}  | after: {huber_loss(y_true_out, y_pred_out):.2f}  (Huber more robust than MSE)")

REGRESSION LOSSES -- House Price Prediction
  y_true: [250. 300. 180. 420. 350. 280. 190. 500.]
  y_pred: [270. 290. 200. 400. 380. 260. 210. 480.]
  Errors: [ 20. -10.  20. -20.  30. -20.  20. -20.]

  MSE:        425.0000  (penalizes large errors heavily)
  RMSE:       20.6155  (interpretable: avg error in same units)
  MAE:        20.0000  (avg absolute error, robust to outliers)
  R-squared:  0.9603  (1.0 = perfect, 0.0 = predicts mean)
  Huber:      19.5000

Outlier effect (one prediction off by 350):
  MSE  before: 425.00  | after: 15687.50  (MSE very sensitive!)
  MAE  before: 20.00  | after: 61.25  (MAE robust)
  RMSE before: 20.62  | after: 125.25
  Huber before:19.50  | after: 60.75  (Huber more robust than MSE)


In [18]:
# --------------------------------------------------
# CLASSIFICATION LOSS FUNCTIONS from scratch
# Binary Cross Entropy, Categorical Cross Entropy
# --------------------------------------------------

def binary_cross_entropy(y_true, y_pred, epsilon=1e-7):
    """
    Binary Cross Entropy (Log Loss).
    Formula: -(1/n) * sum(y*log(p) + (1-y)*log(1-p))

    Where:
      y = true binary label (0 or 1)
      p = predicted probability (0 to 1)

    Properties:
      - Used with sigmoid output (binary classification)
      - Penalizes confident wrong predictions very heavily
      - Range: [0, +inf), perfect model = 0
      - epsilon clips to avoid log(0) = -inf
    """
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)  # numerical stability
    return -np.mean(
        y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
    )

def bce_gradient(y_true, y_pred, epsilon=1e-7):
    """
    Gradient of BCE w.r.t. y_pred.
    dL/dp = -(y/p) + (1-y)/(1-p)
    """
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -(y_true / y_pred) + (1 - y_true) / (1 - y_pred)

def categorical_cross_entropy(y_true_onehot, y_pred_probs, epsilon=1e-7):
    """
    Categorical Cross Entropy.
    Formula: -(1/n) * sum(y_onehot * log(y_pred))

    Where:
      y_onehot    = one-hot encoded labels, shape (n, C)
      y_pred_probs = softmax probabilities,  shape (n, C)

    Used with softmax output (multi-class classification).
    """
    y_pred_probs = np.clip(y_pred_probs, epsilon, 1 - epsilon)
    return -np.mean(np.sum(y_true_onehot * np.log(y_pred_probs), axis=1))

def binary_accuracy(y_true, y_pred, threshold=0.5):
    """Binary classification accuracy."""
    predictions = (y_pred >= threshold).astype(int)
    return np.mean(predictions == y_true)


# --- Test BCE ---
y_true_bin = np.array([1, 0, 1, 1, 0, 0, 1, 0])          # true labels
y_pred_good = np.array([0.92, 0.08, 0.88, 0.95, 0.05, 0.12, 0.91, 0.03])  # good preds
y_pred_bad  = np.array([0.08, 0.92, 0.12, 0.05, 0.95, 0.88, 0.09, 0.97])  # wrong preds

print("=" * 55)
print("BINARY CROSS ENTROPY")
print("=" * 55)
print(f"y_true:       {y_true_bin}")
print(f"y_pred_good:  {y_pred_good}")
print(f"y_pred_bad:   {y_pred_bad}")
print()
print(f"  BCE (good preds): {binary_cross_entropy(y_true_bin, y_pred_good):.6f}  <-- low loss")
print(f"  BCE (bad  preds): {binary_cross_entropy(y_true_bin, y_pred_bad):.6f}  <-- high loss")
print(f"  Accuracy (good):  {binary_accuracy(y_true_bin, y_pred_good):.4f}")
print(f"  Accuracy (bad):   {binary_accuracy(y_true_bin, y_pred_bad):.4f}")

# Loss for individual predictions
print("\nPer-sample BCE:")
print(f"  {'y_true':>8} {'y_pred':>8} {'loss':>12} {'interpretation'}")
print("-" * 55)
test_cases = [
    (1, 0.99), (1, 0.9), (1, 0.5), (1, 0.1), (1, 0.01),
    (0, 0.01), (0, 0.1), (0, 0.5), (0, 0.9), (0, 0.99),
]
for yt, yp in test_cases:
    loss = binary_cross_entropy(np.array([yt]), np.array([yp]))
    note = "near perfect" if loss < 0.1 else ("ok" if loss < 1.0 else "VERY WRONG")
    print(f"  {yt:>8} {yp:>8.2f} {loss:>12.4f}   {note}")

BINARY CROSS ENTROPY
y_true:       [1 0 1 1 0 0 1 0]
y_pred_good:  [0.92 0.08 0.88 0.95 0.05 0.12 0.91 0.03]
y_pred_bad:   [0.08 0.92 0.12 0.05 0.95 0.88 0.09 0.97]

  BCE (good preds): 0.081223  <-- low loss
  BCE (bad  preds): 2.649744  <-- high loss
  Accuracy (good):  1.0000
  Accuracy (bad):   0.0000

Per-sample BCE:
    y_true   y_pred         loss interpretation
-------------------------------------------------------
         1     0.99       0.0101   near perfect
         1     0.90       0.1054   ok
         1     0.50       0.6931   ok
         1     0.10       2.3026   VERY WRONG
         1     0.01       4.6052   VERY WRONG
         0     0.01       0.0101   near perfect
         0     0.10       0.1054   ok
         0     0.50       0.6931   ok
         0     0.90       2.3026   VERY WRONG
         0     0.99       4.6052   VERY WRONG


In [19]:
# --- Categorical Cross Entropy ---

# 3-class classification: cat, dog, bird
# one-hot: cat=[1,0,0], dog=[0,1,0], bird=[0,0,1]
y_true_cat = np.array([
    [1, 0, 0],   # cat
    [0, 1, 0],   # dog
    [0, 0, 1],   # bird
    [1, 0, 0],   # cat
])

y_pred_cat = np.array([
    [0.85, 0.10, 0.05],  # predicts cat (correct)
    [0.20, 0.70, 0.10],  # predicts dog (correct)
    [0.05, 0.15, 0.80],  # predicts bird (correct)
    [0.20, 0.70, 0.10],  # predicts dog (WRONG! true=cat)
])

print("=" * 55)
print("CATEGORICAL CROSS ENTROPY -- 3-class")
print("=" * 55)
print(f"  CCE: {categorical_cross_entropy(y_true_cat, y_pred_cat):.6f}")

# Per-sample
eps = 1e-7
print("\nPer-sample CCE:")
labels = ["cat", "dog", "bird", "cat"]
for i in range(len(y_true_cat)):
    yt = y_true_cat[i]
    yp = np.clip(y_pred_cat[i], eps, 1-eps)
    loss = -np.sum(yt * np.log(yp))
    correct = np.argmax(yt) == np.argmax(yp)
    print(f"  Sample {i+1} (true={labels[i]}): preds={yp}  loss={loss:.4f}  correct={correct}")

CATEGORICAL CROSS ENTROPY -- 3-class
  CCE: 0.587944

Per-sample CCE:
  Sample 1 (true=cat): preds=[0.85 0.1  0.05]  loss=0.1625  correct=True
  Sample 2 (true=dog): preds=[0.2 0.7 0.1]  loss=0.3567  correct=True
  Sample 3 (true=bird): preds=[0.05 0.15 0.8 ]  loss=0.2231  correct=True
  Sample 4 (true=cat): preds=[0.2 0.7 0.1]  loss=1.6094  correct=False


---
# UNIT 11 -- Statistics Functions
## mean, std, var, argmin, argmax, percentile, corrcoef, cumsum

In [20]:
# --------------------------------------------------
# CENTRAL TENDENCY & DISPERSION
# mean, median, std, var, min, max, range
# --------------------------------------------------

data = np.array([12, 45, 23, 67, 34, 89, 45, 23, 78, 56,
                 34, 12, 90, 45, 67, 23, 78, 34, 56, 89])
print("Dataset:", data)
print(f"Size: {data.size}")
print()

# Central tendency
print("--- Central Tendency ---")
print(f"  Mean:       {np.mean(data):.4f}      (sum / count)")
print(f"  Median:     {np.median(data):.4f}      (middle value, robust to outliers)")
print(f"  Weighted mean: {np.average(data, weights=np.ones_like(data)):.4f}")

# Dispersion
print("\n--- Dispersion ---")
print(f"  Std (ddof=0):  {np.std(data):.4f}   (population std)")
print(f"  Std (ddof=1):  {np.std(data, ddof=1):.4f}   (sample std, unbiased)")
print(f"  Variance:      {np.var(data):.4f}   (std^2)")
print(f"  Min:           {np.min(data)}")
print(f"  Max:           {np.max(data)}")
print(f"  Range:         {np.max(data) - np.min(data)}")
print(f"  Peak-to-peak:  {np.ptp(data)}       (same as range)")

# 2D axis-wise stats
mat = np.array([[1, 5, 3, 7],
                [9, 2, 6, 4],
                [8, 3, 5, 1]])
print("\n--- 2D Axis-wise Stats ---")
print("Matrix:")
print(mat)
print(f"\nMean (overall):     {np.mean(mat):.4f}")
print(f"Mean (axis=0, col): {np.mean(mat, axis=0)}  (per column)")
print(f"Mean (axis=1, row): {np.mean(mat, axis=1)}  (per row)")
print(f"Sum  (axis=0):      {np.sum(mat, axis=0)}")
print(f"Sum  (axis=1):      {np.sum(mat, axis=1)}")
print(f"Std  (axis=0):      {np.round(np.std(mat, axis=0), 3)}")
print(f"Max  (axis=1):      {np.max(mat, axis=1)}")  

Dataset: [12 45 23 67 34 89 45 23 78 56 34 12 90 45 67 23 78 34 56 89]
Size: 20

--- Central Tendency ---
  Mean:       50.0000      (sum / count)
  Median:     45.0000      (middle value, robust to outliers)
  Weighted mean: 50.0000

--- Dispersion ---
  Std (ddof=0):  25.2765   (population std)
  Std (ddof=1):  25.9331   (sample std, unbiased)
  Variance:      638.9000   (std^2)
  Min:           12
  Max:           90
  Range:         78
  Peak-to-peak:  78       (same as range)

--- 2D Axis-wise Stats ---
Matrix:
[[1 5 3 7]
 [9 2 6 4]
 [8 3 5 1]]

Mean (overall):     4.5000
Mean (axis=0, col): [6.         3.33333333 4.66666667 4.        ]  (per column)
Mean (axis=1, row): [4.   5.25 4.25]  (per row)
Sum  (axis=0):      [18 10 14 12]
Sum  (axis=1):      [16 21 17]
Std  (axis=0):      [3.559 1.247 1.247 2.449]
Max  (axis=1):      [7 9 8]


In [21]:
# --------------------------------------------------
# argmin, argmax -- INDEX of min/max value return karta hai
# AI use: predicted class = argmax(softmax output)
# --------------------------------------------------

scores = np.array([0.05, 0.12, 0.72, 0.08, 0.03])  # softmax output
classes = ["airplane", "car", "bird", "cat", "deer"]

print("=" * 50)
print("argmin and argmax")
print("=" * 50)
print(f"Scores: {scores}")
print(f"Classes: {classes}")
print()
print(f"  np.argmax(scores) = {np.argmax(scores)}  --> predicted class: '{classes[np.argmax(scores)]}'")
print(f"  np.argmin(scores) = {np.argmin(scores)}  --> least likely class: '{classes[np.argmin(scores)]}'")
print(f"  max prob: {np.max(scores)}  min prob: {np.min(scores)}")

# 2D argmax -- along axis
mat = np.array([[3, 7, 1],
                [9, 2, 8],
                [4, 6, 5]])
print("\n2D argmin/argmax:")
print("Matrix:")
print(mat)
print(f"  argmax (flat):      {np.argmax(mat)}       (linear index 3 = row1,col0 = value 9)")
print(f"  argmin (flat):      {np.argmin(mat)}       (linear index 2 = row0,col2 = value 1)")
print(f"  argmax (axis=0):    {np.argmax(mat, axis=0)}   (row index of max in each col)")
print(f"  argmax (axis=1):    {np.argmax(mat, axis=1)}   (col index of max in each row)")
print(f"  argmin (axis=0):    {np.argmin(mat, axis=0)}")
print(f"  argmin (axis=1):    {np.argmin(mat, axis=1)}")

# unravel_index -- linear index to multi-dim
flat_idx = np.argmax(mat)
row, col = np.unravel_index(flat_idx, mat.shape)
print(f"\n  np.unravel_index({flat_idx}, {mat.shape}) = ({row}, {col}) = value {mat[row, col]}")

# argsort -- sorted indices
vals = np.array([50, 10, 40, 20, 30])
sorted_idx = np.argsort(vals)
print(f"\n  np.argsort([50,10,40,20,30]) = {sorted_idx}")
print(f"  Sorted values: {vals[sorted_idx]}")
print(f"  Top-3 indices: {np.argsort(vals)[-3:][::-1]}  (descending)")

argmin and argmax
Scores: [0.05 0.12 0.72 0.08 0.03]
Classes: ['airplane', 'car', 'bird', 'cat', 'deer']

  np.argmax(scores) = 2  --> predicted class: 'bird'
  np.argmin(scores) = 4  --> least likely class: 'deer'
  max prob: 0.72  min prob: 0.03

2D argmin/argmax:
Matrix:
[[3 7 1]
 [9 2 8]
 [4 6 5]]
  argmax (flat):      3       (linear index 3 = row1,col0 = value 9)
  argmin (flat):      2       (linear index 2 = row0,col2 = value 1)
  argmax (axis=0):    [1 0 1]   (row index of max in each col)
  argmax (axis=1):    [1 0 1]   (col index of max in each row)
  argmin (axis=0):    [0 1 0]
  argmin (axis=1):    [2 1 0]

  np.unravel_index(3, (3, 3)) = (1, 0) = value 9

  np.argsort([50,10,40,20,30]) = [1 3 4 2 0]
  Sorted values: [10 20 30 40 50]
  Top-3 indices: [0 2 4]  (descending)


In [22]:
# --------------------------------------------------
# PERCENTILE, QUANTILE, HISTOGRAM
# AI use: data profiling, outlier detection, EDA
# --------------------------------------------------

np.random.seed(0)
data = np.concatenate([
    np.random.normal(50, 10, 90),   # normal cluster
    np.array([5, 8, 95, 99, 100])   # outliers
])

print("=" * 50)
print("PERCENTILE and QUANTILE")
print("=" * 50)
print(f"  Data size: {len(data)}")
print(f"  Mean: {np.mean(data):.2f}   Std: {np.std(data):.2f}")

# np.percentile
p_vals = [0, 10, 25, 50, 75, 90, 95, 99, 100]
print("\nPercentiles:")
for p in p_vals:
    val = np.percentile(data, p)
    print(f"  P{p:3d}: {val:.2f}")

# IQR -- Interquartile Range (outlier detection)
Q1 = np.percentile(data, 25)
Q3 = np.percentile(data, 75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers = data[(data < lower_fence) | (data > upper_fence)]
print(f"\nIQR Outlier Detection:")
print(f"  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
print(f"  Lower fence: {lower_fence:.2f}")
print(f"  Upper fence: {upper_fence:.2f}")
print(f"  Outliers ({len(outliers)}): {np.round(outliers, 2)}")

# np.quantile -- same as percentile but 0-1 scale
print(f"\nnp.quantile([0.25, 0.5, 0.75]): {np.round(np.quantile(data, [0.25, 0.5, 0.75]), 2)}")

# np.histogram -- frequency distribution
counts, bin_edges = np.histogram(data, bins=8)
print("\nHistogram (8 bins):")
for i in range(len(counts)):
    bar = '#' * int(counts[i] / 2)
    print(f"  [{bin_edges[i]:5.1f} - {bin_edges[i+1]:5.1f}]: {bar} ({counts[i]})")

PERCENTILE and QUANTILE
  Data size: 95
  Mean: 50.66   Std: 14.58

Percentiles:
  P  0: 5.00
  P 10: 35.59
  P 25: 42.22
  P 50: 50.52
  P 75: 57.69
  P 90: 65.17
  P 95: 70.38
  P 99: 99.06
  P100: 100.00

IQR Outlier Detection:
  Q1=42.22, Q3=57.69, IQR=15.47
  Lower fence: 19.02
  Upper fence: 80.90
  Outliers (5): [  5.   8.  95.  99. 100.]

np.quantile([0.25, 0.5, 0.75]): [42.22 50.52 57.69]

Histogram (8 bins):
  [  5.0 -  16.9]: # (2)
  [ 16.9 -  28.8]:  (1)
  [ 28.8 -  40.6]: ####### (15)
  [ 40.6 -  52.5]: ################### (38)
  [ 52.5 -  64.4]: ############ (25)
  [ 64.4 -  76.2]: ##### (11)
  [ 76.2 -  88.1]:  (0)
  [ 88.1 - 100.0]: # (3)


In [23]:
# --------------------------------------------------
# np.corrcoef() -- Correlation Matrix
# AI use: feature selection, EDA before training
# --------------------------------------------------

np.random.seed(42)
n = 50

# Create correlated features
age         = np.random.randint(22, 60, n).astype(float)
experience  = age - 22 + np.random.randn(n) * 2         # correlated with age
salary      = experience * 8000 + np.random.randn(n) * 5000 + 30000  # correlated with exp
height      = np.random.normal(170, 10, n)               # uncorrelated
random_feat = np.random.randn(n)                         # pure noise

# Correlation matrix
features = np.array([age, experience, salary, height, random_feat])
feat_names = ["age", "experience", "salary", "height", "random"]
corr_matrix = np.corrcoef(features)

print("=" * 65)
print("CORRELATION MATRIX")
print("=" * 65)
print("  Range: -1.0 (perfect negative) to +1.0 (perfect positive)")
print("  0.0 = no linear correlation")
print()
print(f"  {'':>12}", end="")
for name in feat_names:
    print(f"  {name:>10}", end="")
print()
print("  " + "-" * 67)
for i, name in enumerate(feat_names):
    print(f"  {name:>12}", end="")
    for j in range(len(feat_names)):
        val = corr_matrix[i, j]
        marker = " ***" if i != j and abs(val) > 0.7 else (" *  " if i != j and abs(val) > 0.3 else "    ")
        print(f"  {val:>8.3f}", end="")
    print()
print("  (* mild, *** strong)")

# Feature selection using correlation
target_corr = corr_matrix[2]  # salary row
print(f"\nCorrelation with salary:")
for i, name in enumerate(feat_names):
    bar = '#' * int(abs(target_corr[i]) * 20)
    print(f"  {name:>12}: {target_corr[i]:>7.3f}  {bar}")

# np.cov -- Covariance Matrix
cov_2feat = np.cov(age, salary)
print(f"\nCovariance (age, salary): {cov_2feat[0,1]:.2f}")
print("  (positive means they move together)")

CORRELATION MATRIX
  Range: -1.0 (perfect negative) to +1.0 (perfect positive)
  0.0 = no linear correlation

                       age  experience      salary      height      random
  -------------------------------------------------------------------
           age     1.000     0.980     0.978     0.056    -0.017
    experience     0.980     1.000     0.998     0.080     0.009
        salary     0.978     0.998     1.000     0.092     0.009
        height     0.056     0.080     0.092     1.000    -0.124
        random    -0.017     0.009     0.009    -0.124     1.000
  (* mild, *** strong)

Correlation with salary:
           age:   0.978  ###################
    experience:   0.998  ###################
        salary:   1.000  ####################
        height:   0.092  #
        random:   0.009  

Covariance (age, salary): 926415.47
  (positive means they move together)


In [24]:
# --------------------------------------------------
# np.cumsum(), np.cumprod(), np.diff(), np.gradient()
# AI use: loss curve tracking, running stats, feature engineering
# --------------------------------------------------

arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

print("=" * 50)
print("CUMULATIVE FUNCTIONS")
print("=" * 50)
print(f"Array: {arr}")
print()

cs = np.cumsum(arr)
cp = np.cumprod(arr[:6])   # only 6 to avoid huge numbers

print(f"np.cumsum():      {cs}")
print(f"np.cumprod()[:6]: {cp}")

# 2D cumsum
mat = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("\n2D cumsum:")
print("Original:")
print(mat)
print("cumsum(axis=0) -- running sum down each column:")
print(np.cumsum(mat, axis=0))
print("cumsum(axis=1) -- running sum across each row:")
print(np.cumsum(mat, axis=1))

# Real AI use: training loss curve running average
losses = np.array([2.3, 1.8, 1.5, 1.3, 1.1, 0.95, 0.88, 0.82, 0.79, 0.75])
cumsum_losses = np.cumsum(losses)
steps = np.arange(1, len(losses) + 1)
running_avg = cumsum_losses / steps

print("\nTraining loss running average:")
print(f"  Epoch losses:   {losses}")
print(f"  Running average: {np.round(running_avg, 3)}")

# np.diff() -- differences between consecutive elements
print("\nnp.diff() -- consecutive differences:")
print(f"  diff(losses):      {np.round(np.diff(losses), 3)}")
print(f"  (negative = loss is decreasing = good)")
print(f"  diff order=2:      {np.round(np.diff(losses, n=2), 3)}")

# np.gradient() -- numerical gradient (like derivative)
x = np.linspace(0, 10, 11)
y = x ** 2   # y = x^2, derivative = 2x
grad = np.gradient(y, x)
print("\nnp.gradient(x^2) -- should be ~2x:")
print(f"  x:        {x}")
print(f"  y=x^2:    {y}")
print(f"  gradient: {grad}  (should be 2x = 0,2,4,6,8,10,12,14,16,18,20)")

CUMULATIVE FUNCTIONS
Array: [ 1  2  3  4  5  6  7  8  9 10]

np.cumsum():      [ 1  3  6 10 15 21 28 36 45 55]
np.cumprod()[:6]: [  1   2   6  24 120 720]

2D cumsum:
Original:
[[1 2 3]
 [4 5 6]
 [7 8 9]]
cumsum(axis=0) -- running sum down each column:
[[ 1  2  3]
 [ 5  7  9]
 [12 15 18]]
cumsum(axis=1) -- running sum across each row:
[[ 1  3  6]
 [ 4  9 15]
 [ 7 15 24]]

Training loss running average:
  Epoch losses:   [2.3  1.8  1.5  1.3  1.1  0.95 0.88 0.82 0.79 0.75]
  Running average: [2.3   2.05  1.867 1.725 1.6   1.492 1.404 1.331 1.271 1.219]

np.diff() -- consecutive differences:
  diff(losses):      [-0.5  -0.3  -0.2  -0.2  -0.15 -0.07 -0.06 -0.03 -0.04]
  (negative = loss is decreasing = good)
  diff order=2:      [ 0.2   0.1   0.    0.05  0.08  0.01  0.03 -0.01]

np.gradient(x^2) -- should be ~2x:
  x:        [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10.]
  y=x^2:    [  0.   1.   4.   9.  16.  25.  36.  49.  64.  81. 100.]
  gradient: [ 1.  2.  4.  6.  8. 10. 12. 14. 16. 18.

---
# UNIT 12 -- Broadcasting

Broadcasting = NumPy ka rule jisse different shaped arrays pe math karna possible hota hai **bina extra memory copy kiye**.

## The 3 Broadcasting Rules (must memorize)

**Rule 1:** If arrays have different number of dimensions, the shape of the smaller array is padded with 1s on the LEFT side.

**Rule 2:** Arrays with size 1 along a dimension are stretched to match the other array's size along that dimension.

**Rule 3:** If sizes still don't match and neither is 1, it raises a ValueError.

In [25]:
# --------------------------------------------------
# BROADCASTING -- Rule 1, 2, 3 step by step
# --------------------------------------------------

print("=" * 65)
print("BROADCASTING -- 3 Rules Explained")
print("=" * 65)

# --- CASE 1: Scalar + Array ---
# Shape: ()  and  (5,)
# Rule 1: scalar becomes (1,)  then Rule 2: (1,) -> (5,)
arr = np.array([1, 2, 3, 4, 5])
result = arr + 10
print("CASE 1: Scalar + 1D array")
print(f"  Shapes: () + {arr.shape}")
print(f"  Steps: () -> (1,) -> (5,)  [Rule 1 + Rule 2]")
print(f"  {arr} + 10 = {result}")

# --- CASE 2: 1D + 2D ---
# Shape: (3,) and (4, 3)
# Rule 1: (3,) becomes (1, 3)
# Rule 2: (1, 3) becomes (4, 3) -- stretched along axis 0
row_vec = np.array([10, 20, 30])        # shape (3,)
matrix  = np.arange(12).reshape(4, 3)   # shape (4, 3)
result2 = matrix + row_vec
print("\nCASE 2: 1D row vector + 2D matrix")
print(f"  Shapes: {row_vec.shape} + {matrix.shape}")
print(f"  Steps: (3,) -> (1,3) -> (4,3)  [Rule 1 + Rule 2]")
print(f"  row_vec: {row_vec}")
print(f"  matrix:\n{matrix}")
print(f"  matrix + row_vec (adds row to EVERY row):\n{result2}")

# --- CASE 3: Column vector + Row vector ---
# Shape: (4, 1) and (1, 3) --> (4, 3)
# Both dimensions expand!
col = np.array([[1], [2], [3], [4]])   # shape (4, 1)
row = np.array([[10, 20, 30]])          # shape (1, 3)
result3 = col + row
print("\nCASE 3: Column vector + Row vector")
print(f"  Shapes: {col.shape} + {row.shape}")
print(f"  Steps: (4,1) + (1,3) -> both stretch to (4,3)  [Rule 2 both dims]")
print(f"  col: {col.ravel()}  row: {row.ravel()}")
print(f"  outer sum result (4x3):\n{result3}")

BROADCASTING -- 3 Rules Explained
CASE 1: Scalar + 1D array
  Shapes: () + (5,)
  Steps: () -> (1,) -> (5,)  [Rule 1 + Rule 2]
  [1 2 3 4 5] + 10 = [11 12 13 14 15]

CASE 2: 1D row vector + 2D matrix
  Shapes: (3,) + (4, 3)
  Steps: (3,) -> (1,3) -> (4,3)  [Rule 1 + Rule 2]
  row_vec: [10 20 30]
  matrix:
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
  matrix + row_vec (adds row to EVERY row):
[[10 21 32]
 [13 24 35]
 [16 27 38]
 [19 30 41]]

CASE 3: Column vector + Row vector
  Shapes: (4, 1) + (1, 3)
  Steps: (4,1) + (1,3) -> both stretch to (4,3)  [Rule 2 both dims]
  col: [1 2 3 4]  row: [10 20 30]
  outer sum result (4x3):
[[11 21 31]
 [12 22 32]
 [13 23 33]
 [14 24 34]]


In [26]:
# --- CASE 4: Incompatible shapes -- ValueError ---

print("CASE 4: Incompatible shapes -- broadcast fails")
a = np.array([1, 2, 3])     # (3,)
b = np.array([1, 2, 3, 4])  # (4,)
try:
    result = a + b
except ValueError as e:
    print(f"  ValueError: {e}")
    print(f"  Rule 3: sizes 3 and 4 don't match, neither is 1 -> FAILS")

# --- CASE 5: 3D broadcasting ---
# Shape: (2, 1, 4) + (1, 3, 1) --> (2, 3, 4)
a = np.ones((2, 1, 4))    # (2, 1, 4)
b = np.ones((1, 3, 1))    # (1, 3, 1)
result = a + b
print("\nCASE 5: 3D Broadcasting")
print(f"  {a.shape} + {b.shape} --> {result.shape}")
print(f"  dim 0: 2 vs 1 -> 2")
print(f"  dim 1: 1 vs 3 -> 3")
print(f"  dim 2: 4 vs 1 -> 4")

# --- CASE 6: Real AI -- batch normalization style ---
# batch: (32, 128) -- 32 samples, 128 features
# mean:  (128,)    -- per-feature mean
# std:   (128,)    -- per-feature std
np.random.seed(0)
batch = np.random.randn(32, 128)  # (32, 128)
feat_mean = batch.mean(axis=0)    # (128,)
feat_std  = batch.std(axis=0)     # (128,)

# Broadcasting: (32, 128) - (128,) --> (128,) expands to (1, 128) --> (32, 128)
normalized = (batch - feat_mean) / (feat_std + 1e-8)

print(f"\nBatch Normalization via Broadcasting:")
print(f"  batch:     {batch.shape}")
print(f"  feat_mean: {feat_mean.shape}")
print(f"  batch - feat_mean: {batch.shape}  (broadcasting!)")
print(f"  normalized: {normalized.shape}")
print(f"  Verify -- per-feature mean after norm: {np.abs(normalized.mean(axis=0)).mean():.8f} (near 0)")
print(f"  Verify -- per-feature std  after norm: {normalized.std(axis=0).mean():.6f} (near 1)")

CASE 4: Incompatible shapes -- broadcast fails
  ValueError: operands could not be broadcast together with shapes (3,) (4,) 
  Rule 3: sizes 3 and 4 don't match, neither is 1 -> FAILS

CASE 5: 3D Broadcasting
  (2, 1, 4) + (1, 3, 1) --> (2, 3, 4)
  dim 0: 2 vs 1 -> 2
  dim 1: 1 vs 3 -> 3
  dim 2: 4 vs 1 -> 4

Batch Normalization via Broadcasting:
  batch:     (32, 128)
  feat_mean: (128,)
  batch - feat_mean: (32, 128)  (broadcasting!)
  normalized: (32, 128)
  Verify -- per-feature mean after norm: 0.00000000 (near 0)
  Verify -- per-feature std  after norm: 1.000000 (near 1)


In [27]:
# --------------------------------------------------
# BROADCASTING COMPATIBILITY CHECK
# np.broadcast_shapes() -- which shapes are compatible?
# --------------------------------------------------

print("Broadcasting Compatibility Reference:")
print(f"{'Shape A':<18} {'Shape B':<18} {'Result Shape':<18} {'Valid?'}")
print("-" * 65)

test_shapes = [
    ((5,),      (5,)),          # exact match
    ((1,),      (5,)),          # 1 stretches to 5
    ((3, 4),    (4,)),           # (4,) -> (1,4) -> (3,4)
    ((3, 4),    (3, 1)),         # col stretches
    ((1, 4),    (3, 1)),         # both stretch -> (3, 4)
    ((2, 3, 4), (3, 4)),         # (3,4) -> (1,3,4) -> (2,3,4)
    ((2, 3, 4), (1, 1, 4)),      # two dims stretch
    ((3,),      (4,)),           # FAIL: 3 != 4, neither is 1
    ((2, 3),    (3, 2)),         # FAIL
    ((5, 1, 4), (1, 3, 1)),      # -> (5, 3, 4)
]

for a_shape, b_shape in test_shapes:
    try:
        result_shape = np.broadcast_shapes(a_shape, b_shape)
        print(f"  {str(a_shape):<18} {str(b_shape):<18} {str(result_shape):<18} OK")
    except ValueError:
        print(f"  {str(a_shape):<18} {str(b_shape):<18} {'INCOMPATIBLE':<18} FAIL")

# np.broadcast_to -- manually broadcast an array
small = np.array([1, 2, 3])
big   = np.broadcast_to(small, (4, 3))  # no copy, just view
print(f"\nnp.broadcast_to({small}, (4,3)):")
print(big)
print(f"  .base is not None (view, no copy): {big.base is not None}")

Broadcasting Compatibility Reference:
Shape A            Shape B            Result Shape       Valid?
-----------------------------------------------------------------
  (5,)               (5,)               (5,)               OK
  (1,)               (5,)               (5,)               OK
  (3, 4)             (4,)               (3, 4)             OK
  (3, 4)             (3, 1)             (3, 4)             OK
  (1, 4)             (3, 1)             (3, 4)             OK
  (2, 3, 4)          (3, 4)             (2, 3, 4)          OK
  (2, 3, 4)          (1, 1, 4)          (2, 3, 4)          OK
  (3,)               (4,)               INCOMPATIBLE       FAIL
  (2, 3)             (3, 2)             INCOMPATIBLE       FAIL
  (5, 1, 4)          (1, 3, 1)          (5, 3, 4)          OK

np.broadcast_to([1 2 3], (4,3)):
[[1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]]
  .base is not None (view, no copy): True


In [28]:
# --------------------------------------------------
# REAL AI BROADCASTING EXAMPLES
# --------------------------------------------------

np.random.seed(42)

print("=" * 60)
print("BROADCASTING IN AI PIPELINES")
print("=" * 60)

# 1. Cosine Similarity for RAG retrieval
# Given: query embedding and document embeddings
query_emb = np.random.randn(1, 512)    # (1, 512) -- one query
doc_embs  = np.random.randn(100, 512)  # (100, 512) -- 100 documents

# Normalize (L2 norm) using broadcasting
query_norm = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
doc_norm   = doc_embs / np.linalg.norm(doc_embs, axis=1, keepdims=True)

# Cosine similarity = dot product of normalized vectors
# query_norm: (1, 512), doc_norm.T: (512, 100) --> (1, 100)
cosine_sims = (query_norm @ doc_norm.T).flatten()  # (100,)

top5_idx = np.argsort(cosine_sims)[-5:][::-1]
print("1. Cosine Similarity for RAG:")
print(f"   query: {query_emb.shape}, docs: {doc_embs.shape}")
print(f"   similarities: {cosine_sims.shape}")
print(f"   Top-5 doc indices: {top5_idx}")
print(f"   Top-5 scores:      {np.round(cosine_sims[top5_idx], 4)}")

# 2. Min-Max Normalization via broadcasting
features = np.random.randn(1000, 20)  # 1000 samples, 20 features
f_min = features.min(axis=0, keepdims=True)   # (1, 20)
f_max = features.max(axis=0, keepdims=True)   # (1, 20)
normalized = (features - f_min) / (f_max - f_min + 1e-8)  # (1000, 20)
print(f"\n2. Feature Normalization:")
print(f"   features: {features.shape}, f_min: {f_min.shape}")
print(f"   (features - f_min): {(features-f_min).shape}  (broadcasting!)")
print(f"   normalized min: {normalized.min():.4f}  max: {normalized.max():.4f}")

# 3. Class weight bias addition (like bias in neural net)
# weight: (output_features=4, input_features=10)
# bias:   (output_features=4,)
# input:  (batch=32, input_features=10)
W     = np.random.randn(4, 10)   # weight matrix
b     = np.random.randn(4)       # bias vector
x_in  = np.random.randn(32, 10)  # batch input

# Linear layer: x @ W.T + b
# x_in @ W.T : (32, 10) @ (10, 4) = (32, 4)
# + b        : (32, 4) + (4,) = (32, 4)  <-- broadcasting!
linear_out = x_in @ W.T + b
print(f"\n3. Linear Layer (Dense):")
print(f"   x: {x_in.shape}, W: {W.shape}, b: {b.shape}")
print(f"   x @ W.T: (32,10) @ (10,4) = {(x_in @ W.T).shape}")
print(f"   + b (broadcasting): {(x_in @ W.T).shape} + {b.shape} = {linear_out.shape}")

BROADCASTING IN AI PIPELINES
1. Cosine Similarity for RAG:
   query: (1, 512), docs: (100, 512)
   similarities: (100,)
   Top-5 doc indices: [39 58 57 48 50]
   Top-5 scores:      [0.1163 0.1059 0.0916 0.0775 0.0746]

2. Feature Normalization:
   features: (1000, 20), f_min: (1, 20)
   (features - f_min): (1000, 20)  (broadcasting!)
   normalized min: 0.0000  max: 1.0000

3. Linear Layer (Dense):
   x: (32, 10), W: (4, 10), b: (4,)
   x @ W.T: (32,10) @ (10,4) = (32, 4)
   + b (broadcasting): (32, 4) + (4,) = (32, 4)


---
# Day 9 Capstone -- NumPy ML Pipeline from Scratch

Sabhi concepts ek saath: reshaping, broadcasting, math, stats, loss functions.

In [29]:
# --------------------------------------------------
# CAPSTONE: Logistic Regression from scratch using NumPy only
# No sklearn, no torch -- pure math
# Concepts used: broadcasting, sigmoid, BCE, gradient descent
# --------------------------------------------------

np.random.seed(42)

# --- Dataset: Binary Classification ---
# Predict pass/fail based on study_hours and sleep_hours
n_samples = 200

# Class 0: fail -- low study, random sleep
X0 = np.column_stack([
    np.random.normal(2, 1, n_samples//2),    # study_hours low
    np.random.normal(6, 1.5, n_samples//2),  # sleep_hours
])
# Class 1: pass -- high study
X1 = np.column_stack([
    np.random.normal(7, 1, n_samples//2),    # study_hours high
    np.random.normal(7, 1.5, n_samples//2),  # sleep_hours
])

X = np.vstack([X0, X1])                       # (200, 2)
y = np.hstack([np.zeros(n_samples//2), np.ones(n_samples//2)])  # (200,)

# Shuffle
idx = np.random.permutation(n_samples)
X, y = X[idx], y[idx]

# Train/test split
split = int(0.8 * n_samples)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Feature normalization (broadcasting)
X_mean = X_train.mean(axis=0)
X_std  = X_train.std(axis=0)
X_train_norm = (X_train - X_mean) / X_std   # broadcasting!
X_test_norm  = (X_test  - X_mean) / X_std

print("=" * 55)
print("LOGISTIC REGRESSION FROM SCRATCH")
print("=" * 55)
print(f"  X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"  y_train: {y_train.shape}  y_test: {y_test.shape}")
print(f"  Class balance: {int(y_train.sum())}/{len(y_train)} positive")

# --- Model: Logistic Regression ---
# f(x) = sigmoid(W @ x + b)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def predict_proba(X, W, b):
    """Forward pass: X(n,f) @ W(f,) + b --> sigmoid --> (n,)"""
    logits = X @ W + b   # (n, f) @ (f,) = (n,)  broadcasting!
    return sigmoid(logits)

def compute_loss(y_true, y_pred, epsilon=1e-7):
    """Binary Cross Entropy"""
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def compute_gradients(X, y_true, y_pred):
    """dL/dW and dL/db -- BCE gradient"""
    n = len(y_true)
    error = y_pred - y_true   # (n,)
    dW = (X.T @ error) / n    # (f,)
    db = np.mean(error)        # scalar
    return dW, db

def accuracy(y_true, y_pred_proba, threshold=0.5):
    preds = (y_pred_proba >= threshold).astype(int)
    return np.mean(preds == y_true)

# --- Training loop ---
n_features = X_train_norm.shape[1]
W = np.zeros(n_features)   # initialize weights to 0
b = 0.0
lr = 0.1
n_epochs = 100

history = {'loss': [], 'val_loss': [], 'acc': [], 'val_acc': []}

print(f"\nTraining -- lr={lr}, epochs={n_epochs}")
print("-" * 55)

for epoch in range(1, n_epochs + 1):
    # Forward
    y_pred_train = predict_proba(X_train_norm, W, b)
    loss         = compute_loss(y_train, y_pred_train)

    # Backward
    dW, db = compute_gradients(X_train_norm, y_train, y_pred_train)

    # Update
    W -= lr * dW
    b -= lr * db

    # Validation
    y_pred_val = predict_proba(X_test_norm, W, b)
    val_loss   = compute_loss(y_test, y_pred_val)
    acc        = accuracy(y_train, y_pred_train)
    val_acc    = accuracy(y_test, y_pred_val)

    history['loss'].append(loss)
    history['val_loss'].append(val_loss)
    history['acc'].append(acc)
    history['val_acc'].append(val_acc)

    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d} | loss: {loss:.4f} | val_loss: {val_loss:.4f} "
              f"| acc: {acc:.3f} | val_acc: {val_acc:.3f}")

# --- Final Results ---
print("-" * 55)
print(f"  Final weights: W = {np.round(W, 4)}, b = {b:.4f}")
print(f"  Final train accuracy: {history['acc'][-1]:.4f}")
print(f"  Final test  accuracy: {history['val_acc'][-1]:.4f}")
print(f"  Loss reduced: {history['loss'][0]:.4f} --> {history['loss'][-1]:.4f}")

# Stats on predictions
final_preds = (predict_proba(X_test_norm, W, b) >= 0.5).astype(int)
correct   = np.sum(final_preds == y_test.astype(int))
incorrect = np.sum(final_preds != y_test.astype(int))
print(f"  Correct: {correct}/{len(y_test)}  Incorrect: {incorrect}/{len(y_test)}")

print("\nAll NumPy operations used:")
print("  reshape/stack      --> data assembly")
print("  broadcasting        --> normalization, linear layer")
print("  sigmoid             --> activation function")
print("  log/clip            --> BCE loss")
print("  mean/std            --> normalization stats")
print("  argsort/argmax      --> top-k, accuracy")
print("  cumsum              --> loss history")
print("  boolean indexing    --> predictions")

LOGISTIC REGRESSION FROM SCRATCH
  X_train: (160, 2)  X_test: (40, 2)
  y_train: (160,)  y_test: (40,)
  Class balance: 82/160 positive

Training -- lr=0.1, epochs=100
-------------------------------------------------------
  Epoch   1 | loss: 0.6931 | val_loss: 0.6681 | acc: 0.512 | val_acc: 1.000
  Epoch  10 | loss: 0.5144 | val_loss: 0.4996 | acc: 0.981 | val_acc: 1.000
  Epoch  20 | loss: 0.3982 | val_loss: 0.3892 | acc: 0.981 | val_acc: 1.000
  Epoch  30 | loss: 0.3260 | val_loss: 0.3200 | acc: 0.988 | val_acc: 1.000
  Epoch  40 | loss: 0.2774 | val_loss: 0.2731 | acc: 0.988 | val_acc: 1.000
  Epoch  50 | loss: 0.2425 | val_loss: 0.2393 | acc: 0.988 | val_acc: 1.000
  Epoch  60 | loss: 0.2162 | val_loss: 0.2138 | acc: 0.988 | val_acc: 1.000
  Epoch  70 | loss: 0.1957 | val_loss: 0.1939 | acc: 0.988 | val_acc: 1.000
  Epoch  80 | loss: 0.1793 | val_loss: 0.1779 | acc: 0.988 | val_acc: 1.000
  Epoch  90 | loss: 0.1658 | val_loss: 0.1647 | acc: 0.988 | val_acc: 1.000
  Epoch 100 | lo

---
# Day 9 Summary

| Concept | Key Functions | AI Use |
|---------|--------------|--------|
| Reshape | reshape, flatten, ravel | Tensor shape changes between layers |
| Transpose | .T, transpose(axes) | Attention, weight matrix |
| Expand/Squeeze | expand_dims, squeeze | Batch dim add/remove |
| Combining | concatenate, vstack, hstack, stack | Dataset assembly, batch creation |
| Splitting | split, vsplit, hsplit | Train/val/test split |
| Copy vs View | .copy(), .base | Memory management, safe ops |
| Arithmetic | +, -, *, /, **, np.add etc | All math ops |
| Ufuncs | sqrt, exp, log, clip, abs | Activations, loss numerics |
| Sigmoid | from scratch | Binary classification output |
| MSE/MAE/RMSE | from scratch | Regression loss |
| BCE/CCE | from scratch | Classification loss |
| Stats | mean, std, percentile, corrcoef | EDA, feature engineering |
| argmin/argmax | argsort, unravel_index | Class prediction, top-k |
| cumsum/diff | cumsum, gradient | Running averages, derivatives |
| Broadcasting | 3 rules | Batch ops, normalization, linear layer |
